In [1]:
# -*- coding: utf-8 -*-
"""
SwinIR Model for Image Super-Resolution

This notebook trains and evaluates a SwinIR model for image super-resolution,
and generates predictions for a test dataset. The results are saved as a CSV file
containing image IDs and pixel values.
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from torch.cuda.amp import GradScaler, autocast

# Define SwinIR model class
class SwinIR(nn.Module):
    def __init__(self, img_size=64, window_size=8, embed_dim=60, depths=[6, 6, 6], num_heads=[6, 6, 6], mlp_ratio=2., upscale=4):
        super(SwinIR, self).__init__()
        self.upscale_factor = upscale
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))

        self.conv_initial = nn.Conv2d(3, embed_dim, kernel_size=3, stride=1, padding=1)

        # Defining the layers
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(embed_dim if i == 0 else embed_dim * 2 ** (i - 1), embed_dim * 2 ** i, kernel_size=3, stride=1, padding=1),
                nn.GELU(),
                nn.Conv2d(embed_dim * 2 ** i, embed_dim * 2 ** i, kernel_size=3, stride=1, padding=1)
            ) for i in range(self.num_layers)
        ])

        self.conv_after_body = nn.Conv2d(self.num_features, embed_dim, kernel_size=3, stride=1, padding=1)
        
        # Upsample block
        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * upscale ** 2, kernel_size=3, stride=1, padding=1),
            nn.PixelShuffle(upscale),
            nn.Conv2d(embed_dim, 3, kernel_size=3, stride=1, padding=1)
        )

    def forward(self, x):
        x = self.conv_initial(x)
        for layer in self.layers:
            x = layer(x)
        x = self.conv_after_body(x)
        x = self.upsample(x)
        return x

# Initialize model
model = SwinIR().cuda()

# Define image transformation pipeline
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Custom dataset class for loading low and high-resolution images
class ImageDataset(Dataset):
    def __init__(self, lr_dir, hr_dir=None, transform=None):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.transform = transform
        self.lr_images = os.listdir(lr_dir)
        if hr_dir:
            self.hr_images = os.listdir(hr_dir)

    def __len__(self):
        return len(self.lr_images)

    def __getitem__(self, idx):
        lr_image = Image.open(os.path.join(self.lr_dir, self.lr_images[idx]))
        if self.hr_dir:
            hr_image = Image.open(os.path.join(self.hr_dir, self.hr_images[idx]))
            if self.transform:
                lr_image = self.transform(lr_image)
                hr_image = self.transform(hr_image)
            return lr_image, hr_image, self.lr_images[idx]
        else:
            if self.transform:
                lr_image = self.transform(lr_image)
            return lr_image, self.lr_images[idx]

# Define file paths
train_lr_dir = '/kaggle/input/dlp-jan-2025-nppe-3/archive/train/train/'
train_hr_dir = '/kaggle/input/dlp-jan-2025-nppe-3/archive/train/gt/'
eval_lr_dir = '/kaggle/input/dlp-jan-2025-nppe-3/archive/val/val/'
eval_hr_dir = '/kaggle/input/dlp-jan-2025-nppe-3/archive/val/gt/'
test_lr_dir = '/kaggle/input/dlp-jan-2025-nppe-3/archive/test/'

# Create dataset instances
train_dataset = ImageDataset(train_lr_dir, train_hr_dir, transform=transform)
eval_dataset = ImageDataset(eval_lr_dir, eval_hr_dir, transform=transform)
test_dataset = ImageDataset(test_lr_dir, transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Initialize mixed precision training
scaler = GradScaler()

# Train the model
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for lr_images, hr_images, _ in train_loader:
        lr_images, hr_images = lr_images.cuda(), hr_images.cuda()

        optimizer.zero_grad()

        with autocast():
            outputs = model(lr_images)
            loss = criterion(outputs, hr_images)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss/len(train_loader)}')

    # Validation phase
    model.eval()
    eval_loss = 0.0
    with torch.no_grad():
        for lr_images, hr_images, _ in eval_loader:
            lr_images, hr_images = lr_images.cuda(), hr_images.cuda()
            outputs = model(lr_images)
            loss = criterion(outputs, hr_images)
            eval_loss += loss.item()

    print(f'Evaluation Loss: {eval_loss/len(eval_loader)}')

# Save model predictions on test data
model.eval()
output_dir = '/kaggle/working/test_predictions'
os.makedirs(output_dir, exist_ok=True)

with torch.no_grad():
    for lr_images, filenames in test_loader:
        lr_images = lr_images.cuda()
        outputs = model(lr_images)
        outputs = outputs.cpu()

        for i, output_img in enumerate(outputs):
            output_img = output_img.permute(1, 2, 0).numpy()
            output_img = (output_img * 0.5) + 0.5  # Unnormalize
            output_img = (output_img * 255).astype(np.uint8)
            Image.fromarray(output_img).save(os.path.join(output_dir, filenames[i]))

# Convert images to CSV format for submission
def images_to_csv(folder_path, output_csv):
    data_rows = []
    for filename in os.listdir(folder_path):
        if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            image_path = os.path.join(folder_path, filename)
            image = Image.open(image_path).convert('L')
            image_array = np.array(image).flatten()[::8]
            image_id = filename.split('.')[0].replace('test_', 'gt_')
            data_rows.append([image_id, *image_array])

    column_names = ['ID'] + [f'pixel_{i}' for i in range(len(data_rows[0]) - 1)]
    df = pd.DataFrame(data_rows, columns=column_names)
    df.to_csv(output_csv, index=False)
    print(f'Successfully saved predictions to {output_csv}')

# Save test predictions to CSV
output_csv = '/kaggle/working/submission.csv'
images_to_csv(output_dir, output_csv)

<ipython-input-1-70a11286c50a>:115: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-1-70a11286c50a>:128: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/50], Training Loss: 0.022382231707439255
Evaluation Loss: 0.026486152129720397
Epoch [2/50], Training Loss: 0.004139646871808903
Evaluation Loss: 0.02536011473345223
Epoch [3/50], Training Loss: 0.0029378856249852947
Evaluation Loss: 0.02470331077477825
Epoch [4/50], Training Loss: 0.0023328983031603297
Evaluation Loss: 0.024456867392161
Epoch [5/50], Training Loss: 0.001974505449675481
Evaluation Loss: 0.024428259289420363
Epoch [6/50], Training Loss: 0.0017877734919324088
Evaluation Loss: 0.024310353114756187
Epoch [7/50], Training Loss: 0.0016994191953972897
Evaluation Loss: 0.023998120021241815
Epoch [8/50], Training Loss: 0.0016729675095935187
Evaluation Loss: 0.024036553604945318
Epoch [9/50], Training Loss: 0.0016348337172950864
Evaluation Loss: 0.02373254206031561
Epoch [10/50], Training Loss: 0.0016481117072215956
Evaluation Loss: 0.024152067309217668
Epoch [11/50], Training Loss: 0.0016895178968123824
Evaluation Loss: 0.024184467896485505
Epoch [12/50], Training Loss